# Input Binding Data Exploration

## Purpose: 

{TODO}

## Packages and Options

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import seaborn as sns 
import matplotlib.pyplot as plt
import itertools, glob, scipy.stats, numpy, tanglegram

## Literals

In [2]:
# dictionary to rename the splice junction positions to numbers
splice_junction_position_renaming = {
    "5_left": 1, 
    "5_right": 2, 
    "center_left": 3, 
    "center_right": 4, 
    "3_left": 5, 
    "3_right": 6
}

# path and regular expression to get the necessary data 
binding_data_path_regex = "../output/control_only_cleaned_input_binding_data/*-100-*csv.gz"

# different types of hierarchical clustering linkage methods 
linkage_methods = ["single", "complete", "average"]


## Loading External Datasets

### Load Ayan's Input Data and Subset to Binding Information Only

In [3]:
# store all input binding data per cell line
all_binding_data = {
    "HepG2": dict(),
    "K562": dict()
}

# for each file in the data path and regex pattern 
for file in sorted(glob.glob(binding_data_path_regex)): 
    
    file = file.replace("\\", "/")
    file
    
    # parse out cell line and binding window size used 
    cell_line = file.split("/")[-1].split("-")[0]
    window_size = file.split("/")[-1].split("-")[1]
    
    # read in pandas 
    binding_data = pd.read_csv(
        file, 
        sep=",", 
        compression="gzip", 
    )
    
    # make sure no NaN values in the DataFrame
    assert binding_data.isna().values.any() == False
    
    # subset only to columns that are related to eCLIP binding
    binding_data = binding_data.loc[
         :, binding_data.columns.str.endswith(
                 ("_right", "_left")
             )
     ]
    
    # convert all 0/1 columns to int8 as they don't need 64 bits
    for column in binding_data.columns:
        if binding_data[column].dtype == 'int64':
            binding_data[column] = binding_data[column].astype("int8")

    # make sure every value in the DataFrame is either a 1 (binding) or 0 (not binding)
    assert all([binding_data[column].isin([0,1]).all() for column in binding_data.columns])
    
    # save the data 
    all_binding_data[cell_line][window_size] = binding_data


'../output/control_only_cleaned_input_binding_data/HepG2-100-50-50.csv.gz'

'../output/control_only_cleaned_input_binding_data/K562-100-50-50.csv.gz'

In [4]:
binding_data.head()

,AARS_5_left,AATF_5_left,ABCF1_5_left,ADAT1_5_left,AGGF1_5_left,AKAP1_5_left,AKAP8L_5_left,APEX1_5_left,APOBEC3C_5_left,AQR_5_left,BUD13_5_left,CPEB4_5_left,CPSF6_5_left,CSTF2T_5_left,DDX1_5_left,DDX21_5_left,DDX24_5_left,DDX3X_5_left,DDX42_5_left,DDX43_5_left,DDX47_5_left,DDX51_5_left,DDX52_5_left,DDX55_5_left,DDX6_5_left,DGCR8_5_left,DHX30_5_left,DROSHA_5_left,EEF2_5_left,EFTUD2_5_left,EIF3G_5_left,EIF4E_5_left,EIF4G2_5_left,ELAC2_5_left,ELAVL1_5_left,EWSR1_5_left,EXOSC10_5_left,EXOSC5_5_left,FAM120A_5_left,FASTKD2_5_left,FMR1_5_left,FTO_5_left,FUS_5_left,FXR1_5_left,FXR2_5_left,GARS_5_left,GEMIN5_5_left,GNL3_5_left,GPKOW_5_left,GRWD1_5_left,GTF2F1_5_left,HLTF_5_left,HNRNPA1_5_left,HNRNPC_5_left,HNRNPK_5_left,HNRNPL_5_left,HNRNPM_5_left,HNRNPU_5_left,HNRNPUL1_5_left,IGF2BP1_5_left,IGF2BP2_5_left,ILF3_5_left,KHDRBS1_5_left,KHSRP_5_left,LARP4_5_left,LARP7_5_left,LIN28B_5_left,LSM11_5_left,MATR3_5_left,MBNL1_5_left,METAP2_5_left,METTL1_5_left,MORC2_5_left,MTPAP_5_left,NCBP2_5_left,NIPBL_5_left,NOLC1_5_left,NONO_5_left,NPM1_5_left,NSUN2_5_left,PABPC4_5_left,PCBP1_5_left,PHF6_5_left,PPIL4_5_left,PRPF8_5_left,PTBP1_5_left,PUM1_5_left,PUM2_5_left,PUS1_5_left,QKI_5_left,RBFOX2_5_left,RBM15_5_left,RBM22_5_left,RNF187_5_left,RPS10_5_left,RPS11_5_left,RPS3_5_left,RPS6_5_left,RYBP_5_left,SAFB_5_left,SAFB2_5_left,SBDS_5_left,SDAD1_5_left,SERBP1_5_left,SF3B1_5_left,SF3B4_5_left,SLBP_5_left,SLTM_5_left,SMNDC1_5_left,SND1_5_left,SRSF1_5_left,SRSF7_5_left,SRSF9_5_left,SSB_5_left,SUPV3L1_5_left,TAF15_5_left,TARDBP_5_left,TBRG4_5_left,TIA1_5_left,TRA2A_5_left,TROVE2_5_left,U2AF1_5_left,U2AF2_5_left,UCHL5_5_left,UPF1_5_left,UTP18_5_left,UTP3_5_left,WDR3_5_left,WDR43_5_left,WRN_5_left,XRCC6_5_left,XRN2_5_left,YBX3_5_left,YWHAG_5_left,ZC3H11A_5_left,ZC3H8_5_left,ZNF622_5_left,ZNF800_5_left,ZRANB2_5_left,AARS_5_right,AATF_5_right,ABCF1_5_right,ADAT1_5_right,AGGF1_5_right,AKAP1_5_right,AKAP8L_5_right,APEX1_5_right,APOBEC3C_5_right,AQR_5_right,BUD13_5_right,CPEB4_5_right,CPSF6_5_right,CSTF2T_5_right,DDX1_5_right,DDX21_5_right,DDX24_5_right,DDX3X_5_right,DDX42_5_right,DDX43_5_right,DDX47_5_right,DDX51_5_right,DDX52_5_right,DDX55_5_right,DDX6_5_right,DGCR8_5_right,DHX30_5_right,DROSHA_5_right,EEF2_5_right,EFTUD2_5_right,EIF3G_5_right,EIF4E_5_right,EIF4G2_5_right,ELAC2_5_right,ELAVL1_5_right,EWSR1_5_right,EXOSC10_5_right,EXOSC5_5_right,FAM120A_5_right,FASTKD2_5_right,FMR1_5_right,FTO_5_right,FUS_5_right,FXR1_5_right,FXR2_5_right,GARS_5_right,GEMIN5_5_right,GNL3_5_right,GPKOW_5_right,GRWD1_5_right,GTF2F1_5_right,HLTF_5_right,HNRNPA1_5_right,HNRNPC_5_right,HNRNPK_5_right,HNRNPL_5_right,HNRNPM_5_right,HNRNPU_5_right,HNRNPUL1_5_right,IGF2BP1_5_right,IGF2BP2_5_right,ILF3_5_right,KHDRBS1_5_right,KHSRP_5_right,LARP4_5_right,LARP7_5_right,LIN28B_5_right,LSM11_5_right,MATR3_5_right,MBNL1_5_right,METAP2_5_right,METTL1_5_right,MORC2_5_right,MTPAP_5_right,NCBP2_5_right,NIPBL_5_right,NOLC1_5_right,NONO_5_right,NPM1_5_right,NSUN2_5_right,PABPC4_5_right,PCBP1_5_right,PHF6_5_right,PPIL4_5_right,PRPF8_5_right,PTBP1_5_right,PUM1_5_right,PUM2_5_right,PUS1_5_right,QKI_5_right,RBFOX2_5_right,RBM15_5_right,RBM22_5_right,RNF187_5_right,RPS10_5_right,RPS11_5_right,RPS3_5_right,RPS6_5_right,RYBP_5_right,SAFB_5_right,SAFB2_5_right,SBDS_5_right,SDAD1_5_right,SERBP1_5_right,SF3B1_5_right,SF3B4_5_right,SLBP_5_right,SLTM_5_right,SMNDC1_5_right,SND1_5_right,SRSF1_5_right,SRSF7_5_right,SRSF9_5_right,SSB_5_right,SUPV3L1_5_right,TAF15_5_right,TARDBP_5_right,TBRG4_5_right,TIA1_5_right,TRA2A_5_right,TROVE2_5_right,U2AF1_5_right,U2AF2_5_right,UCHL5_5_right,UPF1_5_right,UTP18_5_right,UTP3_5_right,WDR3_5_right,WDR43_5_right,WRN_5_right,XRCC6_5_right,XRN2_5_right,YBX3_5_right,YWHAG_5_right,ZC3H11A_5_right,ZC3H8_5_right,ZNF622_5_right,ZNF800_5_right,ZRANB2_5_right,AARS_center_left,AATF_center_left,ABCF1_center_left,ADAT1_center_left,AGGF1_center_left,AKAP1_center_left,AKAP8L_center_left,APEX1_center_left,APOBEC3C_center_left,AQR_center_left,BUD13_center_left,CPEB4_center_left,CPSF6_center_lef

In [5]:
"RBP-Position Combinations that are all 0"

# for each cell line and window size 
for cell_line in all_binding_data: 
    for window_size in all_binding_data[cell_line]:
        "{} {}".format(cell_line, window_size)
        
        tmp_df = all_binding_data[cell_line][window_size]
        
        # list to indicate all those RBP-position combinations that are empty 
        empty=[]
        
        # check if there are RBP positions that only have 0's
        for column in tmp_df.columns:
            if (tmp_df[column]==0).all(): 
                empty.append(column)
                
        print(empty)

'RBP-Position Combinations that are all 0'

'HepG2 100'

['DHX30_5_left', 'HNRNPA1_5_left', 'SAFB_5_left', 'TBRG4_5_left', 'DHX30_5_right', 'HNRNPU_5_right', 'POLR2G_5_right', 'TBRG4_5_right', 'DHX30_center_left', 'POLR2G_center_left', 'TBRG4_center_left', 'DHX30_center_right', 'POLR2G_center_right', 'TBRG4_center_right', 'POLR2G_3_left', 'TBRG4_3_left', 'HNRNPA1_3_right', 'POLR2G_3_right', 'TBRG4_3_right', 'XRCC6_3_right']


'K562 100'

['AARS_5_left', 'GNL3_5_left', 'PUS1_5_left', 'SBDS_5_left', 'SUPV3L1_5_left', 'AARS_5_right', 'ABCF1_5_right', 'HNRNPU_5_right', 'PUS1_5_right', 'RPS11_5_right', 'RYBP_5_right', 'SBDS_5_right', 'SF3B1_5_right', 'UTP3_5_right', 'WDR3_5_right', 'GNL3_center_left', 'PUS1_center_left', 'SBDS_center_left', 'AARS_center_right', 'GNL3_center_right', 'PUS1_center_right', 'RPS11_center_right', 'SBDS_center_right', 'SLBP_center_right', 'WDR3_center_right', 'AARS_3_left', 'PUS1_3_left', 'SBDS_3_left', 'UTP3_3_left', 'AARS_3_right', 'GNL3_3_right', 'HNRNPU_3_right', 'PUS1_3_right', 'SBDS_3_right', 'SUPV3L1_3_right', 'UTP3_3_right', 'WDR3_3_right']


### Map `Rec-Y2H RBP-RBP PPI` Data to `eCLIP RBPs`



In [6]:
rbp_ppi_data = pd.read_excel("../../../inputs/RBP-RBP_PPI/lang_et_al_rec-y2h_screening_results.xlsx")
rbp_ppi_data = rbp_ppi_data[rbp_ppi_data["sumIS"] >=7.1]

rbp_ppi_data.index.size
rbp_ppi_data.head()

2416

,Protein A,Protein B,UniProt species A,UniProt species B,UniProt accessions A,UniProt accessions B,UniProt IDs A,UniProt IDs B,Times detected (RIS > 0),avgIS,sumIS,Found in both orientations,H47_avgIS,H47_sumIS,H47_found in both orientations,H47_times detected (RIS > 0),"Biogrid_all direct evidence (Y2H, reconstituted complex, structure)",Biogrid_all,Hippie,HuRI,Known interaction,Protein A has known interactions,Protein B has known interactions,HPA nuclear A,HPA nuclear B,HPA cytoplasmic A,HPA cytoplasmic B,HPA main locations A,HPA additional locations A,HPA main locations B,HPA additional locations B,HPA shared locations,Youn et al. 2018 (PMID 29395067),NanoBRET,NanoBRET MBU,NanoBRET MBU std,ENCODE eCLIP data A,ENCODE eCLIP data B,ENCODE eCLIP binding sites A,ENCODE eCLIP binding sites B,Jaccard index,Cobinding prob p(A|B),Cobinding prob p(B|A),Cobinding prob p(A|B) (≤54 nt),Cobinding prob p(B|A) (≤54 nt),Close binding events (≤54 nt) A vs. B,Fraction of close binding events A vs. B,Fraction of close binding events in random data A vs. B,Ratio of fractions A vs. B,Resampling p-value A vs. B,Resampling Wilcoxon p-value A vs. B,Close binding events (≤54 nt) B vs. A,Fraction of close binding events B vs. A,Fraction of close binding events in random data B vs. A,Ratio of fractions B vs. A,Resampling p-value B vs. A,Resampling Wilcoxon p-value B vs. A
0,CTBP1,RBM14,HUMAN,MOUSE,Q13363,Q8C2Q3,CTBP1_HUMAN,RBM14_MOUSE,10,8.41,17.19,1,NaN,NaN,NaN,NaN,1,1,1,0,1,1,1,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nuclear speckles,NaN,NaN,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RBM14,CPSF6,MOUSE,MOUSE,Q8C2Q3,Q6NVF9,RBM14_MOUSE,CPSF6_MOUSE,11,13.98,16.03,0,NaN,NaN,NaN,NaN,0,0,0,0,0,1,0,1.0,1.0,0.0,0.0,Nuclear speckles,NaN,Nuclear speckles;Nucleoplasm,NaN,Nuclear speckles,0,NaN,NaN,NaN,0,1,0,791,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PTBP1,PTBP2,HUMAN,HUMAN,P26599,Q9UKA9,PTBP1_HUMAN,PTBP2_HUMAN,8,4.49,15.73,1,0.00,5.47,0.0,5.0,0,0,0,0,0,0,0,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nucleoplasm,NaN,Nucleoplasm,0,pos,13.34,0.94,1,0,16175,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SF1,EWSR1,HUMAN,MOUSE,Q15637,Q61545,SF01_HUMAN,EWS_MOUSE,3,3.29,15.50,0,7.17,11.66,0.0,3.0,1,1,1,0,1,1,1,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nucleoplasm,Nucleoli,Nucleoplasm,0,NaN,NaN,NaN,0,1,0,8788,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRSF11,SRPK2,HUMAN,HUMAN,Q05519,P78362,SRS11_HUMAN,SRPK2_HUMAN,10,9.52,14.79,1,2.32,5.06,0.0,5.0,0,1,1,1,1,1,1,1.0,1.0,0.0,1.0,Nuclear speckles,NaN,Cytosol;Nucleoplasm,NaN,NaN,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# get uniprot mapping file and drop the ENSEMBL ID column and remove any duplicate rows 
uniprot_mapping = pd.read_csv("../../../inputs/RBP-RBP_PPI/uniprot_mapping.tsv", sep="\t").drop("Gene stable ID", axis=1).drop_duplicates()

# get dictionary of gene name OR synonym to the UniProt IDs
gene_names = uniprot_mapping.groupby("Gene name")["UniProtKB Gene Name ID"].apply(list).to_dict()
gene_synonyms = uniprot_mapping.groupby("Gene Synonym")["UniProtKB Gene Name ID"].apply(list).to_dict()

In [8]:
eclip_to_uniprot = {}

# for each cell line and window size 
for cell_line in all_binding_data: 
    eclip_to_uniprot[cell_line] = {}
    
    for window_size in all_binding_data[cell_line]: 
        eclip_to_uniprot[cell_line][window_size]= {}
        
        # get all eCLIP RBPs in this window size and cell line 
        df_rbps = all_binding_data[cell_line][window_size].columns.str.split("_").str[0].unique().tolist()
        
        # for each unique RBP, get all possible UniProt IDs for the gene name
        for rbp in df_rbps:
            tmp_uniprot_list = []
            
            if rbp in gene_names: 
                tmp_uniprot_list.extend(gene_names[rbp])
            
            if rbp in gene_synonyms: 
                tmp_uniprot_list.extend(gene_synonyms[rbp])
            
            # save the rbp list 
            eclip_to_uniprot[cell_line][window_size][rbp] = tmp_uniprot_list

In [9]:
# for each of the dictionaries created above, de-duplicate the final lists
for cell_line in eclip_to_uniprot: 
    for window_size in eclip_to_uniprot[cell_line]: 
        for rbp in eclip_to_uniprot[cell_line][window_size]: 
            
            eclip_to_uniprot[cell_line][window_size][rbp] = list(set(eclip_to_uniprot[cell_line][window_size][rbp]))

In [10]:
# invert the dictionary with keys as uniprot IDs and values as gene names or synonyms
for cell_line in eclip_to_uniprot: 
    for window_size in eclip_to_uniprot[cell_line]: 
        
        tmp_inversion_dict = {}
        
        for rbp in eclip_to_uniprot[cell_line][window_size]: 
            for value in eclip_to_uniprot[cell_line][window_size][rbp]: 
                if value in tmp_inversion_dict: 
                    tmp_inversion_dict = tmp_inversion_dict[value].append(rbp)
                else:
                    tmp_inversion_dict[value] = [rbp]
        
        eclip_to_uniprot[cell_line][window_size] = tmp_inversion_dict

In [11]:
# match up the gene names of the interacting RBPs and check for eCLIP
eclip_rbp_ppis = {}

for cell_line in eclip_to_uniprot: 
    eclip_rbp_ppis[cell_line] ={}
    for window_size in eclip_to_uniprot[cell_line]:
        eclip_rbp_ppis[cell_line][window_size] = []
        
        for index, row in rbp_ppi_data.iterrows():
            interactor_A = row["UniProt accessions A"]
            interactor_B = row["UniProt accessions B"]

            if interactor_A in eclip_to_uniprot[cell_line][window_size] and interactor_B in eclip_to_uniprot[cell_line][window_size]: 
                eclip_rbp_ppis[cell_line][window_size].append(
                    [eclip_to_uniprot[cell_line][window_size][interactor_A], eclip_to_uniprot[cell_line][window_size][interactor_B]]
                )
                

## Which RBPs Bind the Most When Considering All 6 Positions?

Shown as percent of possible binding sites that it binds to. 

In [ ]:
# rbps that have the most amount of binding 
# used for subsetting downstream
top_rbps = []

# for each cell line and window size 
for cell_line in all_binding_data:     
    for window_size in all_binding_data[cell_line]: 
        
        # get number of binding sites per each RBP-position combination 
        tmp_num_binding_sites = all_binding_data[cell_line][window_size].sum(axis=0)        
        
        # parse out each unique RBP in the dataset 
        all_rbps = list(set([rbp_position.split("_")[0] for rbp_position in tmp_num_binding_sites.index]))
        
        # create lists to be index and values for new DataFrame 
        # index = "RBP" and value = "number of times RBP bound to any position"
        df_index = []
        total_binding = []
        
        # for each unique rbp 
        for rbp in all_rbps: 
            
            # sum up number of binding sites across all positions for each rbp and 
            # divide by the total possible number of binding sites 
            total_binding.append(
                (
                    (tmp_num_binding_sites[tmp_num_binding_sites.index.str.contains(rbp)].sum()) / (6 * all_binding_data[cell_line][window_size].index.size)
                ) * 100
            )
            # add RBP to index 
            df_index.append(rbp)
        
        # create dataframe with one column and sort RBPs by total number of binding sites 
        tmp_df = pd.DataFrame(total_binding, index=df_index, columns=["% Total Possible Sites Bound"]).sort_values("% Total Possible Sites Bound")
        
        # cutoff value where RBPs need to have more than this number to be included in downstream analyses
        cutoff_value = tmp_df["% Total Possible Sites Bound"].quantile(.5)
        # subset to the RBP names that are above this cutoff
        top_rbps = tmp_df[tmp_df["% Total Possible Sites Bound"]>cutoff_value].index.to_list()
        
        # create horizontal swarmplot
        plt.figure(figsize=(5,3), dpi=200)
        plt.title("% of All Possible Sites Bound By RBP ({}: {} Window)".format(cell_line, window_size), fontsize=10, pad=10)
        sns.swarmplot(tmp_df, orient="v", size=1.5, color="red")
        plt.ylabel("% of All Possible Sites Bound")
        plt.show()

        # create horizontal heatmap and include rotated annotations for number of total binding sites 
        plt.figure(figsize=(30,2), dpi=200)
        plt.title("% of All Possible Sites Bound By RBP ({}: {} Window)".format(cell_line, window_size), fontsize=24, pad=10)
        sns.heatmap(tmp_df.T, linewidth=0.1, linecolor="grey", annot=True, annot_kws={'rotation':90}, cmap="Blues")
        plt.show()

## Subset to the Top 50 Percent of RBPs with the Highest Binding

In [ ]:
# for cell_line in all_binding_data: 
#     for window_size in all_binding_data[cell_line]: 
        
#         tmp_df = all_binding_data[cell_line][window_size].T
#         tmp_df = tmp_df[tmp_df.index.str.split("_").str[0].isin(top_rbps)].T

#         all_binding_data[cell_line][window_size] = tmp_df

## What is the Distribution of Number of Bindings per Each RBP-Position Combination?

Shown as percentage of total possible places it could have bound to (# `SE` events)

In [ ]:
# dict to store number of binding sites for each RBP position combination 
# {cell line: {window size: value}}
num_binding_sites = {}

# for each cell line and window size
for cell_line in all_binding_data: 
    num_binding_sites[cell_line] = {}
    
    for window_size in all_binding_data[cell_line]: 
        
        # sum up the number of binding sites per RBP-Position combination and save to dict 
        tmp_num_binding_sites = ((all_binding_data[cell_line][window_size].sum(axis=0)) / (all_binding_data[cell_line][window_size].index.size))*100  
        num_binding_sites[cell_line][window_size] = tmp_num_binding_sites


#### Swarmplot of all RBP-Position Combinations

In [ ]:
# for each cell line and window size 
for cell_line in num_binding_sites: 
    for window_size in num_binding_sites[cell_line]: 
        
        tmp_num_binding_sites = num_binding_sites[cell_line][window_size]
        
        # create a swarmplot that is horizontal and small point size because there are a lot of points near 0 
        plt.figure(figsize=(10,3), dpi=200)
        
        sns.swarmplot(tmp_num_binding_sites, orient="v", size=1.5, color="red")
        
        _=plt.suptitle("% SE Events Bound per RBP-Position Combination ({}: {} Window)".format(cell_line, window_size), fontsize=14)
        _=plt.ylabel("% SE Events Bound", fontsize=12)
        
        plt.show()

#### Histogram of all RBP-Position combinations

In [ ]:
# max value to shown on histogram 
hist_max_value = 5

# for each cell line and window size 
for cell_line in num_binding_sites: 
    for window_size in num_binding_sites[cell_line]: 
        
        tmp_num_binding_sites = num_binding_sites[cell_line][window_size]
        
        # create 2 subplots 
        fig, ax = plt.subplots(1, 2, figsize=(20,5), dpi=300, sharey=True)
        
        # first histogram plot uses the entirety of the data 
        # due to severe skewdness, the 2nd plot subsets the x axis by a lot 
        sns.histplot(tmp_num_binding_sites, bins=100, ax=ax[0])
        sns.histplot(tmp_num_binding_sites[tmp_num_binding_sites.between(0,hist_max_value)], bins=100, ax=ax[1])
        
        _=plt.suptitle("Distribution: % of All Possible Sites Bound per RBP-Position Combination ({}: {} Window)".format(cell_line, window_size), fontsize=20, y=1.05)
        _=ax[1].set_title("Same as left plot except subset range is [0, {}]".format(hist_max_value), fontsize=14)
        
        _=fig.supxlabel("RBP-Position Combination Binds to % of All Possible Sites", fontsize=16)
        
        plt.show()

#### Heatmap + ClusterMap w/ matrix of dimension (# RBPs, 6)

In [ ]:
rbp_by_position_matrix = {}

# for each cell line and window size 
for cell_line in num_binding_sites: 
    rbp_by_position_matrix[cell_line] = {}
    
    for window_size in num_binding_sites[cell_line]: 
        
        tmp_num_binding_sites = num_binding_sites[cell_line][window_size]
        
        # create a dictionary for this combination of cell line and window size which will be turned into a DataFrame
        # {"rbp": {"position": "value"}}
        num_binding_sites_df = {}
        
        # for each RBP-position combination and number of total binding sites
        for index, value in tmp_num_binding_sites.items(): 
            
            # for each position in a skipped exon 
            for position_string in splice_junction_position_renaming:
                
                # if position in RBP-position combination matches the "position_string"
                if position_string in index: 
                    # get rbp and position 
                    rbp = index.split("_")[0]
                    position = splice_junction_position_renaming[position_string]
                    
                    # if rbp not already in dictionary, then add it 
                    if rbp not in num_binding_sites_df: 
                        num_binding_sites_df[rbp] = {}
                    
                    # separate out the position and RBP into 2 dimensions
                    num_binding_sites_df[rbp][position] = value
                    
                    # end the innermost loop if you've already found the correct position 
                    break
        
        # from dictionary, convert to DataFrame where the rows are a given position and each column is an RBP 
        num_binding_sites_df = pd.DataFrame.from_dict(num_binding_sites_df, orient="columns")
        num_binding_sites_df.shape

        rbp_by_position_matrix[cell_line][window_size] = num_binding_sites_df

In [ ]:
# heatmap max color value 
heatmap_max_color_value = 5

# for each cell line and window size 
for cell_line in num_binding_sites: 
    for window_size in num_binding_sites[cell_line]: 
        
        num_binding_sites_df = rbp_by_position_matrix[cell_line][window_size]
        
        # create two subplots vertically 
        fig, ax =plt.subplots(2, 1, dpi=300, figsize=(40, 10))
        _=plt.suptitle("% of All Possible Sites Bound by RBP-Position Combination ({}: {} Window Size) \nBottom plot max color set to {}".format(cell_line, window_size, heatmap_max_color_value), fontsize=28)
        
        # first subplot is with all the values 
        # second subplot caps the maximum value color due to the outliers dominating the max color 
        plot1= sns.heatmap(num_binding_sites_df, linewidths=0.1, linecolor="gray", ax=ax[0], cmap="Blues")
        plot2= sns.heatmap(num_binding_sites_df, linewidths=0.1, linecolor="gray", vmin=0, vmax=heatmap_max_color_value, ax=ax[1], cmap="Blues")
        
        plot1.tick_params(axis="y", labelsize=22)
        plot2.tick_params(axis="y", labelsize=22)
    
        plt.show()
        
        # now create two clustermaps where the only difference between both of them 
        # is that the first clustermap doesn't have a limit to the max color and the second one does 
        plot=sns.clustermap(num_binding_sites_df, figsize=(40, 6), linewidths=0.1, linecolor="gray", cmap="Blues", row_cluster=False)
        plot.fig.suptitle("% of All Possible Sites Bound by RBP-Position Combination ({}: {} Window Size)".format(cell_line, window_size), fontsize=36,)
        plot.tick_params(axis="y", labelsize=22)

        plot=sns.clustermap(num_binding_sites_df, figsize=(40, 6), linewidths=0.1, linecolor="gray", vmin=0, vmax=heatmap_max_color_value, cmap="Blues", row_cluster=False)
        plot.fig.suptitle("% of All Possible Sites Bound by RBP-Position Combination ({}: {} Window Size) \nMax color set to {}".format(cell_line, window_size, heatmap_max_color_value), fontsize=36, y=1.05)
        plot.tick_params(axis="y", labelsize=22)
        
        plt.show()
        

## What is the Distribution of the % of RBP-Position Combinations Bound per `SE` Event?

In [ ]:
for cell_line in all_binding_data: 
    for window_size in all_binding_data[cell_line]:

        total_binding = all_binding_data[cell_line][window_size].sum(axis=1)
        percent_binding = ((all_binding_data[cell_line][window_size].sum(axis=1))/ len(all_binding_data[cell_line][window_size].columns))*100
                
        fig, ax = plt.subplots(1,2, dpi=200, figsize=(15,4), sharey=True)
        sns.histplot(
            total_binding,
            bins=total_binding.max(), 
            stat="percent",
            ax=ax[0]
        )
        sns.histplot(
            percent_binding,
            bins=total_binding.max(),
            stat="percent",
            ax=ax[1]
        )
        
        _=plt.suptitle("Histogram of Number + % of RBP-Position Combinations Bound \n{}: {} Window Size".format(cell_line, window_size), y=1.05)
        ax[0].set(xlabel="Number of RBP-Position Combinations Bound")  
        ax[1].set(xlabel="% of RBP-Position Combinations Bound")  

        plt.show()

#### There are no events with 0 RBPs Binding!

#### Looks like Ayan has filtered those out. 

## Do RBPs Binding to `SE` Events Prefer Certain Positions?

NOTE: `Z-Scoring` columns that have only "0" as a value means they turn into "NaN" and are masked in certain visualizations.

Each plot below is sorted by one of the 6 positions to look for patterns

In [ ]:
for cell_line in num_binding_sites: 
    for window_size in num_binding_sites[cell_line]: 

        num_binding_sites_df = rbp_by_position_matrix[cell_line][window_size]
        
        # z score normalize the columns (meaning that each RBP's values at 6 positions are z-scored) 
        # NOTE: columns with no variance become NaN
        num_binding_sites_df = num_binding_sites_df.apply(scipy.stats.zscore, axis="index", nan_policy="raise")  
        
        # remove columns that have NaN values after z score transformation 
        num_binding_sites_df = num_binding_sites_df.dropna(axis="columns", how="all")
                                
        # create clustermap but do not cluster by row as we want to see how the RBPs cluster by positions 
        sns.clustermap(
            num_binding_sites_df, 
            figsize=(35, 6), 
            linewidths=0.1, 
            linecolor="gray", 
            cmap="bwr", 
            row_cluster=False, 
            cbar_pos=(0.1, 0.2, 0.05, 0.5), 
        )
        
        _=plt.suptitle("Hierarchical Clustering of RBPs on Per-RBP Num Binding Sites by Position \n{}: {} Window Size".format(cell_line, window_size), fontsize=40, y=1.2)

        plt.show()
        

## How Similar Are the Position Preferences of RBPs Between Cell Lines?

Can answer this by looking at how Hierarchical Clusterings (from above) compare between Cell Lines. 

* Need to look at BOTH:
    * RBPs profiled in both cell lines WHERE:
        * Both RBPs remain after `Z-Score Transformation` as RBPs with all "0" values are removed

In [ ]:
# take the intersection of the RBPs left after Z scores between each cell line for each window size 
both_cell_line_profiled_rbps = {}

# can do this as the window sizes are same for both cell lines 
for window_size in rbp_by_position_matrix["K562"]: 

    similar_rbps = []
    
    for cell_line in ["K562", "HepG2"]: 

        num_binding_sites_df = rbp_by_position_matrix[cell_line][window_size]
        
        # z score normalize the columns (meaning that each RBP's values at 6 positions are z-scored) 
        # NOTE: columns with no variance become NaN
        num_binding_sites_df = num_binding_sites_df.apply(scipy.stats.zscore, axis="index", nan_policy="raise")  
        
        # remove columns that have NaN values after z score transformation 
        num_binding_sites_df = num_binding_sites_df.dropna(axis="columns", how="all")

        similar_rbps.append(set(num_binding_sites_df.columns.unique()))

    both_cell_line_profiled_rbps[window_size] = similar_rbps[0].intersection(similar_rbps[1])

In [ ]:
for window_size in both_cell_line_profiled_rbps: 
    len(both_cell_line_profiled_rbps[window_size])
    print(both_cell_line_profiled_rbps[window_size])

In [ ]:
z_scored_matrices = {}

for window_size in both_cell_line_profiled_rbps: 
    z_scored_matrices[window_size] = {}
    
    for cell_line in ["K562", "HepG2"]: 

        num_binding_sites_df = rbp_by_position_matrix[cell_line][window_size]
          
        # z score normalize the columns (meaning that each RBP's values at 6 positions are z-scored) 
        # NOTE: columns with no variance become NaN
        num_binding_sites_df = num_binding_sites_df.apply(scipy.stats.zscore, axis="index", nan_policy="raise")  
        
        # remove columns that have NaN values after z score transformation 
        num_binding_sites_df = num_binding_sites_df.dropna(axis="columns", how="all")

        # subset only to those columns that are same between both cell lines 

        z_scored_matrices[window_size][cell_line] = num_binding_sites_df[list(both_cell_line_profiled_rbps[window_size])]


In [ ]:
for window_size in z_scored_matrices:

    plt.figure(dpi=500)
    # create clustermap but do not cluster by row as we want to see how the RBPs cluster by positions 
    cluster_map = sns.clustermap(
        z_scored_matrices[window_size]["K562"], 
        figsize=(35, 6), 
        linewidths=0.1, 
        linecolor="gray", 
        cmap="bwr", 
        row_cluster=False, 
        cbar_pos=(1, 0.2, 0.01, 0.5)
    )
    
    _=plt.suptitle("Clustermap on RBP Num Binding Sites by Position (Z-Scored)\n{}: {} Window Size".format("K562", window_size), fontsize=40, y=1.2)

    plt.show()

    k562_label_order = [t.get_text() for t in cluster_map.ax_heatmap.xaxis.get_majorticklabels()]

    hepg2_heatmap = z_scored_matrices[window_size]["HepG2"][k562_label_order]

    plt.figure(dpi=200, figsize=(20,2))

    sns.heatmap(
        hepg2_heatmap,
        linewidths=0.1, 
        linecolor="gray", 
        cmap="bwr", 
    )

    plt.title("Heatmap of HepG2-{} window size matching K562".format(window_size))

    plt.show()

#################################################################################################################################################
# VICE VERSA 

    plt.figure(dpi=500)
    # create clustermap but do not cluster by row as we want to see how the RBPs cluster by positions 
    cluster_map = sns.clustermap(
        z_scored_matrices[window_size]["HepG2"], 
        figsize=(35, 6), 
        linewidths=0.1, 
        linecolor="gray", 
        cmap="bwr", 
        row_cluster=False, 
        cbar_pos=(1, 0.2, 0.01, 0.5)
    )
    
    _=plt.suptitle("Clustermap on RBP Num Binding Sites by Position (Z-Scored)\n{}: {} Window Size".format("HepG2", window_size), fontsize=40, y=1.2)

    plt.show()

    hepg2_label_order = [t.get_text() for t in cluster_map.ax_heatmap.xaxis.get_majorticklabels()]

    k562_heatmap = z_scored_matrices[window_size]["K562"][hepg2_label_order]

    plt.figure(dpi=200, figsize=(20,2))

    sns.heatmap(
        k562_heatmap,
        linewidths=0.1, 
        linecolor="gray", 
        cmap="bwr", 
    )

    plt.title("Heatmap of K562-{} window size matching HepG2".format(window_size))

    plt.show()
        

### `Tanglegrams` to compare hierarchical clusterings 

In [ ]:
k562_position_preference_distances = pd.DataFrame(
    scipy.spatial.distance.squareform(
        scipy.spatial.distance.pdist(k562_heatmap.T, metric="euclidean")
    ),
    index = k562_heatmap.columns, 
    columns = k562_heatmap.columns
)

k562_position_preference_distances.shape
k562_position_preference_distances.head()

hepg2_position_preference_distances = pd.DataFrame(
    scipy.spatial.distance.squareform(
        scipy.spatial.distance.pdist(hepg2_heatmap.T, metric="euclidean")
    ),
    index = hepg2_heatmap.columns, 
    columns = hepg2_heatmap.columns
)

hepg2_position_preference_distances.shape
hepg2_position_preference_distances.head()

In [ ]:
plt.figure(dpi=100)

tanglegram.tanglegram(
    k562_position_preference_distances,
    hepg2_position_preference_distances, 
    sort=True, 

)

plt.show()

## What RBP-Position Combinations Co-Occur The Most In Space?

Answer with Hamming distance between two vectors where each vector is an RBP-position combination

In [ ]:
# dictionary indicating colors to be used in dendogram plot for labeling position colors
position_colors = {
    1: "#000000", 
    2: "#f032e6", 
    3: "#800000", 
    4: "#ffe119", 
    5: "#42d4f4", 
    6: "#e6194B"
}

In [ ]:
hamming_distances = {}

for cell_line in all_binding_data: 
    hamming_distances[cell_line] = {}
    
    for window_size in all_binding_data[cell_line]: 
        "{} {}".format(cell_line, window_size)
        
        tmp_distance = scipy.spatial.distance.pdist(
            all_binding_data[cell_line][window_size].T.to_numpy(), 
            metric="hamming"
        )
        
        hamming_distances[cell_line][window_size] = tmp_distance 
        
        tmp_distance.shape

In [ ]:
clusterings = {}

for cell_line in hamming_distances: 
    clusterings[cell_line] = {}
    for window_size in hamming_distances[cell_line]: 
        clusterings[cell_line][window_size] = {}
                
        for method in linkage_methods: 
            clusterings[cell_line][window_size][method] = scipy.cluster.hierarchy.linkage(
                hamming_distances[cell_line][window_size], 
                method=method
                
            )

In [ ]:
for cell_line in clusterings: 
    for window_size in clusterings[cell_line]: 
            
        "{} {}".format(cell_line, window_size)

        _=plt.figure(figsize=(60,4), dpi=600)

        dendogram=scipy.cluster.hierarchy.dendrogram(
            clusterings[cell_line][window_size]["average"], 
            get_leaves=True,
            labels = all_binding_data[cell_line][window_size].columns, 
            orientation="top"
        )
        
        for label in plt.gca().get_xmajorticklabels(): 
            
            # set color by getting the position as a string 
            # which is used as key to dict to get numerical position
            # which is used as key to get the color associated with that numerical position
            label.set_color(
                position_colors[
                    splice_junction_position_renaming[
                        "_".join(label.get_text().split("_")[1:])
                    ]
                ]
            )       
            
        plt.show()

## Unique Binding Patterns

* How many unique patterns exist?
* What is the distribution of the number of times each unique binding pattern is seen?
* What is the relationship between the number of times a binding pattern is seen and the density of binding to that `SE` event?
* Which binding patterns have PSI values that have the least amount of variance?
    * Only meaningful if the binding pattern is somewhat interesting and "in-silico" KDs mimic the pattern 

In [ ]:
%%capture  
# providing annoying "PerformanceWarnings" that I want to suppress

unique_binding_patterns = {}

for cell_line in all_binding_data: 
    unique_binding_patterns[cell_line] = {}
    for window_size in all_binding_data[cell_line]:

        tmp_df = all_binding_data[cell_line][window_size].value_counts().reset_index()
        assert tmp_df.notna().any().any()

        assert tmp_df.index.size == tmp_df.drop_duplicates().index.size

        unique_binding_patterns[cell_line][window_size] = tmp_df



In [ ]:
for cell_line in unique_binding_patterns: 
    for window_size in unique_binding_patterns[cell_line]: 

        "Number of unique binding patterns for {} {}".format(cell_line, window_size)
        unique_binding_patterns[cell_line][window_size].index.size

In [ ]:
for cell_line in unique_binding_patterns: 
    for window_size in unique_binding_patterns[cell_line]: 

        plt.figure(dpi=200, figsize=(15, 5))

        _=sns.histplot(
            unique_binding_patterns[cell_line][window_size]["count"], 
            bins=[0,10,20,30,40,50,60,70,80,3000]
        )
        _=plt.title("Distribution of Number of Times Each Unique Binding Pattern Is Seen \n{} {}".format(cell_line, window_size))
        _=plt.xlabel("Number of Times Each Unique Binding Pattern Is Seen")
        plt.show()
        

In [ ]:
for cell_line in unique_binding_patterns: 
    for window_size in unique_binding_patterns[cell_line]: 
        tmp_df = unique_binding_patterns[cell_line][window_size]
                
        only_binding = tmp_df.drop("count", axis=1)
        
        binding_per_event = only_binding.sum(axis=1).to_list()
        
        plt.figure(figsize=(15,4), dpi=200)
        
        plt.scatter(tmp_df["count"].to_list(), binding_per_event, s=2)
        
        plt.title("{}-{} Window Size: # Times Unique Binding Pattern Found vs. Number of RBP-Position Combinations Bound".format(cell_line, window_size))
        plt.xlabel("# Times Unique Binding Pattern Found")
        plt.ylabel("Number of RBP-Position Combinations Bound")
        
        plt.show()

In [ ]:
for cell_line in unique_binding_patterns: 
    for window_size in unique_binding_patterns[cell_line]: 
        tmp_df = unique_binding_patterns[cell_line][window_size]
        
        tmp_df = tmp_df[tmp_df["count"]<100]
        
        only_binding = tmp_df.drop("count", axis=1)
        
        binding_per_event = only_binding.sum(axis=1).to_list()
        
        plt.figure(figsize=(15,4), dpi=200)
        
        plt.scatter(tmp_df["count"].to_list(), binding_per_event, s=5)
        
        plt.title("{}-{} Window Size (x-axis < 100)\n# Times Unique Binding Pattern Found vs. Number of RBP-Position Combinations Bound".format(cell_line, window_size))
        plt.xlabel("# Times Unique Binding Pattern Found")
        plt.ylabel("Number of RBP-Position Combinations Bound")
        
        plt.show()

In [ ]:
top_most_common_binding_patterns =10

for cell_line in unique_binding_patterns: 
    for window_size in unique_binding_patterns[cell_line]: 
        tmp_df = unique_binding_patterns[cell_line][window_size].iloc[0:top_most_common_binding_patterns, :]
        
        only_binding = tmp_df.drop("count", axis=1)
        
        plt.figure(figsize=(50,5), dpi=300)
        sns.heatmap(only_binding, linewidth=0.001, linecolor="grey",cmap="Blues")
        plt.title("{} Most Common Binding Patterns ({}: {} Window Size)".format(top_most_common_binding_patterns, cell_line, window_size), fontsize=40, pad=10)

        plt.show()

## Combinatorics of Co-Binding

For n=1, … ,k: 

- How many times do you find `n` RBPs bound to the same splice junction?
    - (e.g.) for `n=3`, how many times does `RBP1`, `RBP2`, and `RBP3` bind at `position_1`

In [ ]:
#### Pseudocode

for cell line: 
    for window size: 
        subset to the binding data matrix for this cell line and window size 
        get all RBPs in matrix 
        
        for choose_number : 
            get all possible RBP combinations
            
            for each rbp combination: 
                
                for each of 6 positions: 
                    
                    for each rbp

                    for each skipped exon event:

                        tmp_list = []
                        for each rbp at that position: 
                            get value of that rbp at that position

                        if all(tmp_list): 
                            counter+=1
                    
                    save number of times all of them are bound 
                
            

In [ ]:
combination_results = {}
choose_numbers = [3]


for cell_line in all_binding_data:
    combination_results[cell_line] = {}
    
    for window_size in all_binding_data[cell_line]: 
        combination_results[cell_line][window_size] = {}
        
        rbps = all_binding_data[cell_line][window_size].columns.str.split("_").str[0].unique().tolist()
        
        for choose_number in choose_numbers: 

            combination_results[cell_line][window_size][choose_number] = {}

            combos = list(itertools.combinations(rbps, choose_number))
            
            for position in splice_junction_position_renaming: 
                
                meta_list = []
                
                for combo in combos: 
                    tmp_list = []
                    
                    for item in combo: 
                        tmp_list.append("_".join([item, position]))
                    
                    meta_list.append(tmp_list)
 
            
                combination_results[cell_line][window_size][choose_number][position] = meta_list

                

In [ ]:
pd.DataFrame(combination_results).head()